# **VoxCeleb2 Inventory Construction Pipeline**

---

## 🎯 **Objective**
This notebook implements a data processing pipeline to construct:
- segment-level metadata for each audio file  
- speaker-level inventory with aggregated statistics  

The pipeline operates on VoxCeleb2 audio aligned with AgeVoxCeleb annotations,
preparing the data for downstream tasks such as dataset construction
and age verification experiments.

---

## 📌 **Outputs**
The notebook generates the following files:
- `voxceleb_segment_metadata.csv` containing segment-level information  
- `voxceleb_speaker_inventory.csv` containing speaker-level statistics and roles  

## **1. Environment Setup**

This section installs required dependencies and mounts Google Drive.

In [ ]:
!pip install datasets soundfile librosa
!apt-get install -y ffmpeg
!pip install tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **2. VoxCeleb Data Download and Setup**

This step prepares the VoxCeleb dataset by navigating to the working directory in Google Drive, downloading the audio archives, extracting the audio files into a local directory, and retrieving the AgeVoxCeleb metadata files (`utt2age.train`, `utt2age.test`).


These resources are required for **building the speaker inventory** and **aligning the audio data with age annotations**.

In [ ]:
# Navigate to working directory in Google Drive
%cd "/content/drive/MyDrive/age verification/data/voxceleb2"

/content/drive/.shortcut-targets-by-id/1bhY8iYqsfkYl4BZwwQrFpOQk4ZtSMe0h/age verification/data/voxceleb


In [ ]:
# Download first part of VoxCeleb dataset
!wget -O vox2_aac_1.zip "https://huggingface.co/datasets/ProgramComputer/voxceleb/resolve/main/vox2/vox2_aac_1.zip"

# Download second part of VoxCeleb dataset
!wget -O vox2_aac_2.zip "https://huggingface.co/datasets/ProgramComputer/voxceleb/resolve/main/vox2/vox2_aac_2.zip"

In [ ]:
# Extract first archive (skip already extracted files if rerun)
!unzip -n vox2_aac_1.zip -d vox2_audio

# Extract second archive
!unzip vox2_aac_2.zip -d vox2_audio

In [ ]:
# Download test age annotations
!wget -O raw_metadata/utt2age.test https://raw.githubusercontent.com/nttcslab-sp/agevoxceleb/master/utt2age.test

# Download train age annotations
!wget -O raw_metadata/utt2age.train https://raw.githubusercontent.com/nttcslab-sp/agevoxceleb/master/utt2age.train

--2026-04-08 19:21:09--  https://raw.githubusercontent.com/nttcslab-sp/agevoxceleb/master/utt2age.test
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 465449 (455K) [text/plain]
Saving to: ‘utt2age.test’

utt2age.test        100%[===================>] 454.54K  --.-KB/s    in 0.04s   

2026-04-08 19:21:09 (10.6 MB/s) - ‘utt2age.test’ saved [465449/465449]

--2026-04-08 19:21:09--  https://raw.githubusercontent.com/nttcslab-sp/agevoxceleb/master/utt2age.train
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4404797 (4.2M) [text/plain

## **3. VoxCeleb Inventory Construction**

This section implements the main processing pipeline for constructing
segment-level metadata and speaker-level inventory from VoxCeleb data.

The pipeline consists of the following steps:
- **Age Annotation Loading:** loading AgeVoxCeleb annotations  
- **File Validation:** validating the availability of audio files  
- **Duration Extraction:** extracting audio durations with resume support  
- **Gender Integration:** attaching speaker gender metadata  
- **Segment Construction:** constructing segment-level metadata  
- **Speaker Aggregation:** aggregating speaker-level statistics  
- **Role Assignment:** assigning speaker roles based on total duration  

The outputs generated in this section are used for dataset construction
and downstream modeling tasks.

In [ ]:
# -*- coding: utf-8 -*-
"""
VoxCeleb Speaker Inventory Builder

This script constructs segment-level metadata and speaker-level inventory
for the VoxCeleb portion of the dataset construction protocol.

Current scope
-------------
This script is designed for the adult speaker pool derived from AgeVoxCeleb
annotations aligned with VoxCeleb audio files.

Main outputs
------------
1. Segment metadata:
   One row per available utterance segment.

2. Speaker inventory:
   One row per speaker, including:
   - dataset_source
   - speaker_id
   - source_age
   - mapped_age
   - gender
   - total_raw_duration_sec
   - num_files

Notes
-----
- This script operates before VAD-based preprocessing and segmentation.
- It uses raw file durations only.
- It is intended for inventory construction, not final dataset generation.
"""

from __future__ import annotations

import os
import random

import subprocess
from tqdm import tqdm
from pathlib import Path
from typing import Optional

import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)


# =========================================================
# Configuration
# =========================================================

AUDIO_ROOT = Path("/content/drive/MyDrive/age verification/data/voxceleb2/vox2_audio")

VOX2_META_CSV = Path("/content/drive/MyDrive/age verification/data/voxceleb2/raw_metadata/vox2_meta.csv")
AGE_TRAIN = Path("/content/drive/MyDrive/age verification/data/voxceleb2/raw_metadata/utt2age.train")
AGE_TEST = Path("/content/drive/MyDrive/age verification/data/voxceleb2/raw_metadata/utt2age.test")

SEGMENT_OUT = Path("/content/drive/MyDrive/age verification/data/voxceleb2/metadata/voxceleb_segment_metadata.csv")
SPEAKER_OUT = Path("/content/drive/MyDrive/age verification/data/voxceleb2/metadata/voxceleb_speaker_inventory.csv")
TEMP_DURATIONS_OUT = Path("/content/drive/MyDrive/age verification/data/voxceleb2/metadata/durations_progress.csv")

DATASET_SOURCE = "voxceleb"
ADULT_AGE_THRESHOLD = 18
MIN_TOTAL_DURATION_SEC = 12.0
SAVE_EVERY = 500


# =========================================================
# Age Label Loading
# =========================================================
def load_agevoxceleb_file(path: Path) -> pd.DataFrame:
    """
    Load an AgeVoxCeleb annotation file.

    Expected input format
    ---------------------
    Each line contains:
        speaker_id/video_id/segment_id age

    Returns
    -------
    pd.DataFrame
        Columns:
        - speaker_id
        - video_id
        - segment_id
        - age
        - file_path
    """
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) != 2:
                continue

            utt_id, age = parts
            utt_parts = utt_id.split("/")

            if len(utt_parts) != 3:
                continue

            speaker_id, video_id, segment_id = utt_parts

            rows.append(
                {
                    "speaker_id": speaker_id.strip(),
                    "video_id": video_id.strip(),
                    "segment_id": segment_id.strip(),
                    "age": float(age),
                    "file_path": f"dev/aac/{speaker_id}/{video_id}/{segment_id}.m4a",
                }
            )

    return pd.DataFrame(rows)


def load_all_age_labels(train_path: Path, test_path: Path) -> pd.DataFrame:
    """
    Load and concatenate AgeVoxCeleb train and test annotations.
    """
    train_df = load_agevoxceleb_file(train_path)
    test_df = load_agevoxceleb_file(test_path)

    age_df = pd.concat([train_df, test_df], ignore_index=True)
    return age_df


# =========================================================
# File Utilities
# =========================================================
def attach_full_paths(age_df: pd.DataFrame, audio_root: Path) -> pd.DataFrame:
    """
    Attach absolute file paths and filter out missing files.
    """
    df = age_df.copy()
    df["full_path"] = df["file_path"].apply(lambda p: str(audio_root / p))
    df["exists"] = df["full_path"].apply(os.path.exists)

    existing_df = df[df["exists"]].copy()
    existing_df.drop(columns=["exists"], inplace=True)

    return existing_df


def get_duration_sec(audio_path: str) -> Optional[float]:
    """
    Read audio duration in seconds using ffprobe.

    Returns
    -------
    float or None
        Duration in seconds if successful, otherwise None.
    """
    try:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            audio_path,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return round(float(result.stdout.strip()), 3)

    except Exception:
        return None


def load_saved_durations(progress_path: Path) -> pd.DataFrame:
    """
    Load previously saved duration progress if available.
    """
    if progress_path.exists():
        durations_df = pd.read_csv(progress_path)
    else:
        durations_df = pd.DataFrame(columns=["full_path", "duration_sec"])

    durations_df["full_path"] = durations_df["full_path"].astype(str)
    return durations_df


def extract_missing_durations(
    segment_df: pd.DataFrame,
    progress_path: Path,
    save_every: int = SAVE_EVERY,
) -> pd.DataFrame:
    """
    Extract durations for files that have not yet been processed.

    Progress is saved incrementally to support resuming interrupted runs.
    """
    durations_df = load_saved_durations(progress_path)
    done_paths = set(durations_df["full_path"].astype(str))

    segment_df = segment_df.copy()
    segment_df["full_path"] = segment_df["full_path"].astype(str)

    to_process = segment_df[~segment_df["full_path"].isin(done_paths)].copy()

    print(f"Already processed durations: {len(done_paths)}")
    print(f"Remaining files to process: {len(to_process)}")

    if len(to_process) == 0:
        return durations_df

    batch_results = []

    for i, path in enumerate(tqdm(to_process["full_path"], desc="Processing audio"), start=1):
        dur = get_duration_sec(path)
        batch_results.append({"full_path": path, "duration_sec": dur})

        if i % save_every == 0:
            batch_df = pd.DataFrame(batch_results)

            if durations_df.empty:
                durations_df = batch_df.copy()
            else:
                durations_df = pd.concat([durations_df, batch_df], ignore_index=True)

            durations_df = durations_df.drop_duplicates(subset=["full_path"], keep="last")
            durations_df.to_csv(progress_path, index=False)
            batch_results = []

    if batch_results:
        batch_df = pd.DataFrame(batch_results)

        if durations_df.empty:
            durations_df = batch_df.copy()
        else:
            durations_df = pd.concat([durations_df, batch_df], ignore_index=True)

        durations_df = durations_df.drop_duplicates(subset=["full_path"], keep="last")
        durations_df.to_csv(progress_path, index=False)

    return durations_df


# =========================================================
# Gender Metadata
# =========================================================
def load_voxceleb_gender_metadata(meta_csv_path: Path) -> pd.DataFrame:
    """
    Load VoxCeleb metadata and standardize speaker_id and gender columns.
    """
    meta_df = pd.read_csv(meta_csv_path)
    meta_df.columns = meta_df.columns.str.strip()

    meta_df = meta_df.rename(
        columns={
            "VoxCeleb2 ID": "speaker_id",
            "Gender": "gender",
        }
    )

    meta_df["speaker_id"] = meta_df["speaker_id"].astype(str).str.strip()
    meta_df["gender"] = meta_df["gender"].astype(str).str.strip()

    return meta_df[["speaker_id", "gender"]].copy()


def attach_gender(segment_df: pd.DataFrame, gender_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge speaker gender into segment metadata.
    """
    df = segment_df.copy()
    df["speaker_id"] = df["speaker_id"].astype(str).str.strip()

    gender_map = dict(zip(gender_df["speaker_id"], gender_df["gender"]))
    df["gender"] = df["speaker_id"].map(gender_map)

    return df


# =========================================================
# Filtering
# =========================================================
def keep_adult_segments(
    segment_df: pd.DataFrame,
    adult_age_threshold: int = ADULT_AGE_THRESHOLD,
) -> pd.DataFrame:
    """
    Keep only adult utterances based on the age threshold.

    Current protocol assumption
    ---------------------------
    VoxCeleb is used here for the adult pool.
    """
    return segment_df[segment_df["age"] > adult_age_threshold].copy()


# =========================================================
# Output Builders
# =========================================================
def build_segment_metadata(
    age_df: pd.DataFrame,
    gender_df: pd.DataFrame,
    durations_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construct segment-level metadata for available VoxCeleb files.
    """
    segment_df = attach_gender(age_df, gender_df)

    durations_df = durations_df.copy()
    durations_df["full_path"] = durations_df["full_path"].astype(str)

    segment_df = segment_df.merge(durations_df, on="full_path", how="left")

    segment_df = keep_adult_segments(segment_df)

    segment_df = segment_df[
        [
            "speaker_id",
            "video_id",
            "segment_id",
            "age",
            "gender",
            "duration_sec",
            "file_path",
            "full_path",
        ]
    ].copy()

    return segment_df


def build_speaker_inventory(segment_df: pd.DataFrame) -> pd.DataFrame:
    """
    Construct speaker-level inventory required by the dataset protocol.

    Output columns
    --------------
    - dataset_source
    - speaker_id
    - source_age
    - mapped_age
    - gender
    - total_raw_duration_sec
    - num_files
    """
    speaker_inventory_df = (
        segment_df
        .groupby("speaker_id", as_index=False)
        .agg(
            source_age=("age", "mean"),
            gender=("gender", "first"),
            total_raw_duration_sec=("duration_sec", "sum"),
            num_files=("file_path", "count"),
        )
    )

    speaker_inventory_df["dataset_source"] = DATASET_SOURCE
    speaker_inventory_df["mapped_age"] = "adult"

    speaker_inventory_df["source_age"] = speaker_inventory_df["source_age"].round().astype(int)
    speaker_inventory_df["total_raw_duration_sec"] = speaker_inventory_df["total_raw_duration_sec"].round(3)

    speaker_inventory_df = speaker_inventory_df[
        [
            "dataset_source",
            "speaker_id",
            "source_age",
            "mapped_age",
            "gender",
            "total_raw_duration_sec",
            "num_files",
        ]
    ].copy()

    return speaker_inventory_df


def filter_inventory_by_duration(
    speaker_inventory_df: pd.DataFrame,
    min_total_duration_sec: float = MIN_TOTAL_DURATION_SEC,
) -> pd.DataFrame:
    """
    Remove speakers with insufficient total raw duration.
    """
    return speaker_inventory_df[
        speaker_inventory_df["total_raw_duration_sec"] >= min_total_duration_sec
    ].copy()


def assign_inventory_role(total_raw_duration_sec: float) -> str:
    """
    Assign preliminary speaker role based on total raw duration.

    Rules
    -----
    < 12 sec   -> drop
    12-90 sec  -> real_candidates
    > 90 sec   -> randomly assign to spoof_targets or backup
    """
    if total_raw_duration_sec < 12:
        return "drop"
    if total_raw_duration_sec <= 90:
        return "real_candidates"

    return random.choice(["spoof_targets", "backup"])


# =========================================================
# Main
# =========================================================
def main() -> None:
    """
    Run the full VoxCeleb inventory construction pipeline.
    """
    print("\n" + "=" * 60)
    print("VOXCELEB INVENTORY PIPELINE")
    print("=" * 60)

    print("[1/7] Loading AgeVoxCeleb labels...")
    age_df = load_all_age_labels(AGE_TRAIN, AGE_TEST)
    total_age_rows = len(age_df)

    print("[2/7] Checking local file availability...")
    age_df = attach_full_paths(age_df, AUDIO_ROOT)
    total_found_files = len(age_df)

    age_df = load_all_age_labels(AGE_TRAIN, AGE_TEST)
    age_df = attach_full_paths(age_df, AUDIO_ROOT)

    print("[3/7] Loading VoxCeleb gender metadata...")
    gender_df = load_voxceleb_gender_metadata(VOX2_META_CSV)

    print("[4/7] Extracting durations with resume support...")
    durations_df = extract_missing_durations(
        age_df,
        TEMP_DURATIONS_OUT,
        save_every=SAVE_EVERY,
    )

    print("[5/7] Building segment metadata...")
    segment_df = build_segment_metadata(age_df, gender_df, durations_df)
    adult_segment_rows = len(segment_df)

    missing_gender = segment_df["gender"].isna().sum()
    missing_duration = segment_df["duration_sec"].isna().sum()

    print("[6/7] Saving segment metadata...")
    SEGMENT_OUT.parent.mkdir(parents=True, exist_ok=True)
    segment_df.to_csv(SEGMENT_OUT, index=False)

    print("[7/7] Building speaker inventory...")
    speaker_inventory_df = build_speaker_inventory(segment_df)
    speaker_inventory_df["role"] = speaker_inventory_df["total_raw_duration_sec"].apply(assign_inventory_role)

    # ===== BEFORE FILTER =====
    role_counts_before = speaker_inventory_df["role"].value_counts(dropna=False)
    role_counts_before.index.name = None

    # ===== FILTER =====
    speaker_inventory_df = filter_inventory_by_duration(
        speaker_inventory_df,
        min_total_duration_sec=MIN_TOTAL_DURATION_SEC,
    )

    print("Saving speaker inventory...")
    speaker_inventory_df.to_csv(SPEAKER_OUT, index=False)

    # ===== AFTER FILTER =====
    gender_counts_after = speaker_inventory_df["gender"].value_counts(dropna=False)
    gender_counts_after.index.name = None

    # ===== SUMMARY =====
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"Total age-labeled rows       : {total_age_rows}")
    print(f"Files found locally          : {total_found_files}")
    print(f"Adult segment rows retained  : {adult_segment_rows}")
    print(f"Missing gender rows          : {missing_gender}")
    print(f"Missing duration rows        : {missing_duration}")
    print(f"Segment metadata saved to    : {SEGMENT_OUT}")
    print(f"Speaker inventory saved to   : {SPEAKER_OUT}")

    # ===== ROLE =====
    print("\n" + "=" * 60)
    print("ROLE DISTRIBUTION")
    print("=" * 60)
    print(role_counts_before.to_string())

    # ===== GENDER =====
    print("\n" + "=" * 60)
    print("GENDER DISTRIBUTION")
    print("=" * 60)
    print(gender_counts_after.to_string())


if __name__ == "__main__":
    main()


VOXCELEB INVENTORY PIPELINE
[1/7] Loading AgeVoxCeleb labels...
[2/7] Checking local file availability...
[3/7] Loading VoxCeleb gender metadata...
[4/7] Extracting durations with resume support...
Already processed durations: 167927
Remaining files to process: 0
[5/7] Building segment metadata...
[6/7] Saving segment metadata...
[7/7] Building speaker inventory...
Saving speaker inventory...

SUMMARY
Total age-labeled rows       : 167940
Files found locally          : 167927
Adult segment rows retained  : 166023
Missing gender rows          : 0
Missing duration rows        : 0
Segment metadata saved to    : /content/drive/MyDrive/age verification/data/voxceleb2/metadata/voxceleb_segment_metadata.csv
Speaker inventory saved to   : /content/drive/MyDrive/age verification/data/voxceleb2/metadata/voxceleb_speaker_inventory.csv

ROLE DISTRIBUTION
real_candidates    1612
backup             1498
spoof_targets      1444
drop                386

GENDER DISTRIBUTION
m    2773
f    1781


## **4. Age Consistency Analysis**

This section identifies speakers whose segments are classified
as both **minor** and **adult** (i.e., appearing in both age groups),
and evaluates whether using the mean age as a representative label is reliable.

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/age verification/data/voxceleb2/metadata/voxceleb_segment_metadata.csv")

In [ ]:
# Group segments by speaker and compute age statistics
# This gives us:
# - min_age: smallest estimated age across segments
# - max_age: largest estimated age across segments
# - mean_age: average age (used later as representative age)
# - age_std: variation of age within the speaker
# - num_segments: number of segments per speaker
speaker_age_range = (
    df.groupby("speaker_id")["age"]
    .agg(["min", "max", "mean", "std", "count"])
    .reset_index()
    .rename(columns={
        "min": "min_age",
        "max": "max_age",
        "mean": "mean_age",
        "std": "age_std",
        "count": "num_segments"
    })
)

# Identify speakers whose age estimates cross the classification boundary
# Condition:
# - min_age < 19  → at least one segment classified as minor (≤18)
# - max_age ≥ 19 → at least one segment classified as adult
# These speakers are considered inconsistent (crossing the threshold)
crossing_speakers = speaker_age_range[
    (speaker_age_range["min_age"] < 19) &
    (speaker_age_range["max_age"] >= 19)
].copy()

# Report the number and percentage of such speakers
# This helps evaluate whether using average age is safe
print("=== SPEAKERS CROSSING AGE THRESHOLD ===")
print(f"Number of speakers: {len(crossing_speakers)}")
print(f"Percentage: {len(crossing_speakers) / len(speaker_age_range) * 100:.2f}%")

=== SPEAKERS CROSSING AGE THRESHOLD ===
Number of speakers: 0
Percentage: 0.00%


In [ ]:
print("\n=== OVERALL STD ANALYSIS ===")
print(f"Average std across speakers: {speaker_age_range['age_std'].mean():.2f}")
print(f"Max std: {speaker_age_range['age_std'].max():.2f}")


=== OVERALL STD ANALYSIS ===
Average std across speakers: 2.10
Max std: 27.14
